# Module 11 Integrated Lab — Evaluating AI Services

**Duration**: 90 minutes · **Day**: 4

This notebook builds a reusable **evaluation harness** for a black-box sentiment analysis API.
Four habits that separate professional AI evaluation from guesswork:

1. **Load → Iterate → Score → Accumulate → Report** — the fixed shape of every eval harness
2. **Pure scorer functions in `lib/`** — tested, reusable, auditable
3. **The Methodology Paragraph in three places** — zero ambiguity about what was measured
4. **Failures surfaced explicitly** — headline accuracy alone is not auditable

The companion script `eval_sentiment.py` is the production harness; this notebook is the
*thinking space* where you build toward it.

## Section 1 — Setup and Orientation (10 min)

The Integration has pre-shipped two components:

| Component | Purpose |
|-----------|---------|
| `sentiment-api/app.py` | FastAPI service — accepts text, returns `positive` / `negative` / `neutral` |
| `data/sentiment_demo.json` | 15 hand-labeled product reviews (the ground truth) |

Your role is to **evaluate** the service, not modify it. This section confirms everything is
running and shows you the response shape before any scoring logic is written.

In [1]:
%pip install fastapi uvicorn httpx pytest --quiet   # install framework, runner, HTTP client, test runner

import sys                                          # manipulate the module search path
import json                                         # parse JSON data files and API responses
import pathlib                                      # object-oriented, OS-agnostic file paths
import subprocess                                   # launch and communicate with external processes
import time                                         # sleep while the server initialises

import httpx                                        # HTTP client — the same library used in the harness

sys.path.insert(0, '.')                             # make lib/ importable from the project root

Note: you may need to restart the kernel to use updated packages.


### Starting the API server

We launch uvicorn as a background subprocess so the server runs while the notebook executes.
`stdout=subprocess.DEVNULL` silences the server access logs — this is cosmetic, not necessary.

> **Tip**: If port 8000 is already in use, change `--port 8000` below and update `BASE_URL` to match.

In [2]:
server = subprocess.Popen(                     # Popen returns immediately; process runs in background
    [sys.executable, '-m', 'uvicorn',          # invoke uvicorn via Python's module runner
     'app:app',                                # app.py exports an object named `app`
     '--port', '8001',                         # listen on port 8001
     '--log-level', 'error'],                  # suppress info / access logs
    cwd='sentiment-api',                       # run from sentiment-api/ so app.py is on the path
    stdout=subprocess.DEVNULL,                 # discard stdout — change to None to see server logs
    stderr=subprocess.DEVNULL,                 # discard stderr — change to None to debug startup errors
)
time.sleep(2)                                  # give uvicorn 2 seconds to bind the port

BASE_URL = 'http://localhost:8001'             # every API call in this notebook uses this constant

print(f'Server PID: {server.pid} — ready at {BASE_URL}')  # confirm the process started

Server PID: 583662 — ready at http://localhost:8001


### Loading the labeled dataset

`data/sentiment_demo.json` is a list of 15 dicts, each with three fields:

- `id` — stable integer identifier
- `text` — raw review string
- `label` — human-assigned ground-truth sentiment (`positive` / `negative` / `neutral`)

In [3]:
DATA_PATH = pathlib.Path('data/sentiment_demo.json')   # path relative to the notebook directory

reviews = json.loads(DATA_PATH.read_text())            # read_text() returns str; json.loads() parses it

print(f'Loaded {len(reviews)} reviews\n')              # confirm the dataset size

print(f'{"id":>3}  {"label":8}  text')                 # header row
print('-' * 60)                                        # separator line
for r in reviews[:5]:                                  # preview the first 5 entries
    rid   = r['id']                                    # extract to a variable — avoids nested quotes
    label = r['label']                                 # ground-truth sentiment label
    text  = r['text'][:45]                             # truncate long text for display
    print(f'{rid:3d}  {label:8s}  {text}')

Loaded 15 reviews

 id  label     text
------------------------------------------------------------
  1  positive  This product is absolutely amazing!
  2  negative  Terrible quality, broke after one day.
  3  positive  Good value for money, very happy with it.
  4  negative  Completely useless. Waste of money.
  5  positive  Exactly what I needed, very happy.


### Your first API call — inspecting the response shape

Before writing evaluation code, call the API once by hand. Goal: understand exactly what fields
come back so the harness can parse them reliably.

In [4]:
sample   = reviews[0]                                  # first review as a test case

response = httpx.post(                                 # send one POST request
    f'{BASE_URL}/predict',                             # the prediction endpoint
    json={'text': sample['text']},                     # the API expects {"text": "..."}
    timeout=5.0,                                       # fail fast if the server is unresponsive
)

payload  = response.json()                             # decode the response body to a dict
pred     = payload['sentiment']                        # the prediction field
truth    = sample['label']                             # the ground-truth label

print('HTTP status :', response.status_code)           # 200 = success
print('Full payload:')                                 # label the pretty-print block
print(json.dumps(payload, indent=2))                   # display all returned fields

print(f'\nGround truth : {truth}')                    # show what the label says
print(f'Predicted    : {pred}')                        # show what the API returned
print(f'Match        : {truth == pred}')               # True if the model got it right

HTTP status : 200
Full payload:
{
  "sentiment": "positive",
  "confidence": 0.8,
  "latency_ms": 20.29
}

Ground truth : positive
Predicted    : positive
Match        : True


## Section 2 — The Load → Iterate → Score → Accumulate → Report Pattern (15 min)

Every evaluation harness has the same five-step skeleton regardless of the task:

```
LOAD data → ITERATE reviews → POST to API → ACCUMULATE results → REPORT
```

> **The harness is a script, not a notebook.** Notebooks are for exploration;
> scripts produce reproducible, diffable output. `eval_sentiment.py` is the
> production script. In this section you see its skeleton before any scoring
> logic is added.

In [5]:
harness_src = pathlib.Path('eval_sentiment.py').read_text()   # read the eval script into a string
print(harness_src)                                            # display the full source code

"""
Sentiment Evaluation Harness — Module 11 Integrated Lab.

Evaluation Methodology:
    This harness evaluates a black-box sentiment classification service
    against a 15-item labeled dataset of product reviews.  Each review is
    submitted via HTTP POST to /predict.  Predictions are collected, then
    scored using accuracy and macro-averaged F1.  A per-item breakdown is
    included in the output report so that individual failures are auditable.
    Latency is measured via the service's own /metrics endpoint (p95).
    The harness is deterministic: given the same dataset and the same service
    state, it will produce the same scores.

Usage:
    python eval_sentiment.py

Prerequisites:
    - The sentiment API must be running on http://localhost:8001
    - Start it with: uvicorn app:app --port 8001 (from sentiment-api/)
"""

import json                                  # parse the labeled-review JSON file
import pathlib                               # OS-agnostic file paths
impo

### Walking each step in isolation

Executing the five steps one at a time makes the structure concrete before they are
compressed into functions.

In [6]:
# ── Step 1: LOAD ──────────────────────────────────────────────────────────
reviews = json.loads(                                         # json.loads parses a str to a Python list
    pathlib.Path('data/sentiment_demo.json').read_text()      # read_text() returns the file as a str
)
print(f'LOAD complete: {len(reviews)} reviews ready')         # confirm the step

LOAD complete: 15 reviews ready


In [7]:
# ── Steps 2–4: ITERATE → POST → ACCUMULATE ────────────────────────────────
results = []                                                  # the accumulator — starts empty

for review in reviews[:3]:                                    # iterate (first 3 for this walkthrough)
    resp = httpx.post(                                        # Step 3: POST the review to /predict
        f'{BASE_URL}/predict',                                # full URL
        json={'text': review['text']},                        # JSON body containing the review text
        timeout=5.0,                                          # give up after 5 seconds
    )
    payload   = resp.json()                                   # decode the JSON response

    rid       = review['id']                                  # review identifier
    label     = review['label']                               # ground-truth label
    predicted = payload['sentiment']                          # model's prediction
    correct   = (label == predicted)                          # True if the model matched the label

    results.append({                                          # Step 4: accumulate one dict per review
        'id':        rid,
        'text':      review['text'],                          # keep text for the failure report
        'label':     label,
        'predicted': predicted,
        'correct':   correct,
    })

    status = 'OK  ' if correct else 'FAIL'                    # visual indicator
    print(f'[{status}] id={rid:2d}  label={label:8s}  pred={predicted}')  # progress line

print(f'\nAccumulated {len(results)} results into the list')  # confirm Step 4 complete

[OK  ] id= 1  label=positive  pred=positive
[OK  ] id= 2  label=negative  pred=negative
[OK  ] id= 3  label=positive  pred=positive

Accumulated 3 results into the list


## Section 3 — The Methodology Paragraph (10 min)

Every evaluation must answer four questions:

1. What data was used?
2. How were predictions collected?
3. What metrics were computed, and how?
4. Is the process deterministic?

**The Evaluation Methodology Rule**: write this paragraph once and copy it *verbatim* into
three places:

| Location | Reason |
|----------|--------|
| Script module docstring | Source of truth — what actually ran |
| Evaluation spec / rubric | What assessors check against |
| Learner-facing guide | What students are told to measure |

Identical wording in all three places eliminates the gap between "what I measured" and
"what the spec required."

In [8]:
METHODOLOGY = (                                              # parentheses allow multi-line concatenation
    'This harness evaluates a black-box sentiment classification service '
    'against a 15-item labeled dataset of product reviews.  Each review is '
    'submitted via HTTP POST to /predict.  Predictions are collected, then '
    'scored using accuracy and macro-averaged F1.  A per-item breakdown is '
    'included in the output report so that individual failures are auditable.  '
    "Latency is measured via the service's own /metrics endpoint (p95).  "
    'The harness is deterministic: given the same dataset and the same service '
    'state, it will produce the same scores.'
)

print('=== Evaluation Methodology ===')                     # section header
print()
print(METHODOLOGY)                                          # display the full paragraph
print()
print('This exact paragraph appears in:')                   # remind students of the three-place rule
print('  1. eval_sentiment.py  (module docstring)')          # place 1: script
print('  2. the evaluation spec  (rubric)')                  # place 2: spec
print('  3. the learner-facing guide')                       # place 3: guide

=== Evaluation Methodology ===

This harness evaluates a black-box sentiment classification service against a 15-item labeled dataset of product reviews.  Each review is submitted via HTTP POST to /predict.  Predictions are collected, then scored using accuracy and macro-averaged F1.  A per-item breakdown is included in the output report so that individual failures are auditable.  Latency is measured via the service's own /metrics endpoint (p95).  The harness is deterministic: given the same dataset and the same service state, it will produce the same scores.

This exact paragraph appears in:
  1. eval_sentiment.py  (module docstring)
  2. the evaluation spec  (rubric)
  3. the learner-facing guide


## Section 4 — Scorer Functions Live in `lib/` (10 min)

Scoring logic belongs in **pure functions** — functions that accept only their arguments,
return a value, and have no side effects (no HTTP calls, no file I/O, no global state).

| Property | Why it matters |
|----------|---------------|
| **Testable** | A synthetic unit test takes 3 lines — no server needed |
| **Reusable** | Any harness can `import` them |
| **Auditable** | The math lives in one visible place |

The pattern:
```
compute = pure function in lib/
harness = thin coordinator that calls lib/ functions
```

In [9]:
from lib.scorers import accuracy, macro_f1               # import the pure scoring functions

# ── accuracy demo ─────────────────────────────────────────────────────────
y_true_a = ['positive', 'negative', 'negative', 'neutral']  # synthetic ground truth (4 items)
y_pred_a = ['positive', 'positive', 'negative', 'neutral']  # 1 wrong: negative predicted as positive

acc_demo = accuracy(y_true_a, y_pred_a)                     # compute: 3 correct / 4 total

print('accuracy() demo')
print(f'  y_true : {y_true_a}')                             # show inputs
print(f'  y_pred : {y_pred_a}')
print(f'  result : {acc_demo:.4f}   (3 correct / 4 total = 0.75)')   # explain the arithmetic
print()

help(accuracy)                                              # display the function's docstring

accuracy() demo
  y_true : ['positive', 'negative', 'negative', 'neutral']
  y_pred : ['positive', 'positive', 'negative', 'neutral']
  result : 0.7500   (3 correct / 4 total = 0.75)

Help on function accuracy in module lib.scorers:

accuracy(y_true: Sequence[str], y_pred: Sequence[str]) -> float
    Return the fraction of predictions that exactly match the ground-truth label.



In [10]:
# ── macro_f1 demo + synthetic unit test ───────────────────────────────────
# This is the same synthetic input used in tests/test_scorers.py::test_macro_f1_one_wrong

y_true_f = ['positive', 'negative', 'neutral']             # one example of each class
y_pred_f = ['positive', 'positive', 'neutral']             # 'negative' misclassified as 'positive'

f1_demo = macro_f1(y_true_f, y_pred_f)                    # compute macro-averaged F1

print('macro_f1() demo')
print(f'  y_true  : {y_true_f}')                           # show inputs
print(f'  y_pred  : {y_pred_f}')
print(f'  result  : {f1_demo:.4f}   (expected ≈ 0.5556)')  # known value
print()
print('Per-class breakdown:')
print('  positive : precision=1/2=0.500  recall=1/1=1.000  F1=0.667')  # manual calculation
print('  negative : precision=0/0=0.000  recall=0/1=0.000  F1=0.000')  # no TPs
print('  neutral  : precision=1/1=1.000  recall=1/1=1.000  F1=1.000')  # perfect
print('  macro avg: (0.667 + 0.000 + 1.000) / 3 ≈ 0.556')              # average
print()

# Replicate the assertion from the unit test
assert 0.50 < f1_demo < 0.70, 'unexpected result — check the implementation'  # same bounds as test
print('Assertion passed — result matches tests/test_scorers.py expectation')   # confirm

macro_f1() demo
  y_true  : ['positive', 'negative', 'neutral']
  y_pred  : ['positive', 'positive', 'neutral']
  result  : 0.5556   (expected ≈ 0.5556)

Per-class breakdown:
  positive : precision=1/2=0.500  recall=1/1=1.000  F1=0.667
  negative : precision=0/0=0.000  recall=0/1=0.000  F1=0.000
  neutral  : precision=1/1=1.000  recall=1/1=1.000  F1=1.000
  macro avg: (0.667 + 0.000 + 1.000) / 3 ≈ 0.556

Assertion passed — result matches tests/test_scorers.py expectation


In [11]:
# ── Run the full test suite ───────────────────────────────────────────────
result = subprocess.run(                                   # run pytest as a child process
    [sys.executable, '-m', 'pytest', 'tests/', '-v'],      # -v: show each test name individually
    capture_output=True,                                   # capture stdout and stderr
    text=True,                                             # decode bytes to str automatically
)
print(result.stdout)                                       # display the test results
if result.returncode != 0:                                 # non-zero means at least one test failed
    print('STDERR:', result.stderr)                        # surface any error messages

============================= test session starts ==============================
platform linux -- Python 3.13.14, pytest-9.0.2, pluggy-1.6.0 -- /usr/bin/python3.13
cachedir: .pytest_cache
rootdir: /home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 11 Integrated lab
plugins: langsmith-0.8.9, anyio-4.12.1, cov-7.1.0
collecting ... collected 8 items

tests/test_scorers.py::test_accuracy_all_correct PASSED                  [ 12%]
tests/test_scorers.py::test_accuracy_one_wrong PASSED                    [ 25%]
tests/test_scorers.py::test_accuracy_all_wrong PASSED                    [ 37%]
tests/test_scorers.py::test_accuracy_length_mismatch_raises PASSED       [ 50%]
tests/test_scorers.py::test_accuracy_empty_raises PASSED                 [ 62%]
tests/test_scorers.py::test_macro_f1_perfect PASSED                      [ 75%]
tests/test_scorers.py::test_macro_f1_one_wrong PASSED                    [ 87%]
tests/test_scorers.py::test_macro_f1_binary PASSED                       [1

## Section 5 — Reading `/metrics` for Derived Signals (10 min)

The sentiment service exposes a `/metrics` endpoint that reports runtime telemetry.
The Integration has pre-written `lib/metrics_reader.py` to fetch and parse it.
**You use this helper; you do not write it.**

### What is p95 latency?

**p95 (95th-percentile latency)** means: *95% of requests completed faster than this value.*

It is more honest than the mean because it captures **tail latency** — the slow requests that
real users actually experience. A system with mean = 20 ms but p95 = 900 ms has a serious
problem that the mean hides entirely.

In [12]:
# ── Hit /metrics directly to see the raw shape ────────────────────────────
metrics_raw = httpx.get(f'{BASE_URL}/metrics').json()    # GET /metrics and decode the JSON body
print('Raw /metrics payload:')                           # label the block
print(json.dumps(metrics_raw, indent=2))                 # pretty-print the full dict

Raw /metrics payload:
{
  "request_count": 5,
  "p95_latency_ms": 20.29
}


In [13]:
# ── Use the pre-shipped helper ────────────────────────────────────────────
from lib.metrics_reader import fetch_metrics, extract_p95   # import the Integration-supplied helper

metrics = fetch_metrics(BASE_URL)                            # fetch /metrics (with error checking)
p95     = extract_p95(metrics)                              # pull out the p95 value (may be None)

if p95 is not None:                                          # guard: None means no requests served yet
    print(f'p95 latency : {p95:.1f} ms')                    # display rounded to one decimal
    print()
    print('Interpretation:')
    print('  95% of requests to this service completed faster than the value above.')
    print('  This number will appear in the Headline Metrics table of the report.')
else:
    print('p95 not yet available — no requests have been served')   # explain the null case

p95 latency : 20.3 ms

Interpretation:
  95% of requests to this service completed faster than the value above.
  This number will appear in the Headline Metrics table of the report.


## Section 6 — Writing the Report: Surface the Failures (15 min)

Headline accuracy on this dataset: **86.67%** — but that single number hides critical information.

Two rules for honest evaluation reporting:

1. **A headline without a per-item breakdown is not auditable.**
   You cannot tell *which* predictions are wrong, so you cannot fix them.

2. **A per-item breakdown without surfaced failures is not honest.**
   Burying two failures in a 15-row table is functionally the same as hiding them.

The report must do **both**: state the headline metric *and* call out every failure explicitly
with its ID, true label, predicted label, and verbatim text.

In [14]:
# ── Run the production harness ────────────────────────────────────────────
run = subprocess.run(                                      # execute eval_sentiment.py as a child process
    [sys.executable, 'eval_sentiment.py'],                 # the full production harness
    capture_output=True,                                   # capture all output
    text=True,                                             # decode bytes to str
)
print(run.stdout)                                          # display per-item results + final metrics
if run.returncode != 0:                                    # non-zero = script raised an exception
    print('Error:', run.stderr)                            # surface the traceback

Evaluating 15 reviews against http://localhost:8001

[✓] id= 1  label=positive  pred=positive  conf=0.80
[✓] id= 2  label=negative  pred=negative  conf=0.80
[✓] id= 3  label=positive  pred=positive  conf=0.90
[✓] id= 4  label=negative  pred=negative  conf=0.90
[✓] id= 5  label=positive  pred=positive  conf=0.80
[✓] id= 6  label=negative  pred=negative  conf=0.90
[✓] id= 7  label=positive  pred=positive  conf=0.90
[✓] id= 8  label=negative  pred=negative  conf=0.90
[✓] id= 9  label=neutral   pred=neutral   conf=0.60
[✓] id=10  label=positive  pred=positive  conf=0.90
[✗] id=11  label=negative  pred=positive  conf=0.61
[✓] id=12  label=positive  pred=positive  conf=0.80
[✗] id=13  label=negative  pred=positive  conf=0.55
[✓] id=14  label=positive  pred=positive  conf=0.90
[✓] id=15  label=neutral   pred=neutral   conf=0.60

Accuracy   : 86.67%
Macro F1   : 0.8917
p95 Latency: 22.9 ms

Report written → report.md



In [15]:
# ── Display the generated Markdown report ────────────────────────────────
report_text = pathlib.Path('report.md').read_text()        # read the report file
print(report_text)                                         # display raw markdown (open in viewer to render)

# Sentiment Evaluation Report

## Evaluation Methodology

This harness evaluates a black-box sentiment classification service against a 15-item labeled dataset of product reviews.  Each review is submitted via HTTP POST to /predict.  Predictions are collected, then scored using accuracy and macro-averaged F1.  A per-item breakdown is included in the output report so that individual failures are auditable.  Latency is measured via the service's own /metrics endpoint (p95).  The harness is deterministic: given the same dataset and the same service state, it will produce the same scores.

## Headline Metrics

| Metric | Value |
|--------|-------|
| Accuracy   | 86.67% |
| Macro F1   | 0.8917 |
| p95 Latency | 22.9 ms |

## Per-Item Results

| ID | Label | Predicted | Correct | Text (truncated) |
|----|-------|-----------|---------|------------------|
|  1 | positive | positive | ✓ | This product is absolutely amazing! |
|  2 | negative | negative | ✓ | Terrible quality, broke after one da

In [16]:
# ── Reproduce the eval inline and spotlight the failures ─────────────────
all_results = []                                           # fresh accumulator

for review in reviews:                                     # iterate all 15 reviews
    resp      = httpx.post(                                # POST the review text
        f'{BASE_URL}/predict',
        json={'text': review['text']},                     # request body
        timeout=5.0,
    )
    payload   = resp.json()                                # decode the response
    label     = review['label']                            # ground-truth label
    predicted = payload['sentiment']                       # model's prediction

    all_results.append({                                   # accumulate one result per review
        'id':        review['id'],
        'text':      review['text'],
        'label':     label,
        'predicted': predicted,
        'correct':   (label == predicted),                 # boolean match
    })

# ── Compute headline metrics ──────────────────────────────────────────────
y_true    = [r['label']     for r in all_results]          # ground-truth label sequence
y_pred    = [r['predicted'] for r in all_results]          # prediction sequence
acc       = accuracy(y_true, y_pred)                       # overall accuracy
f1        = macro_f1(y_true, y_pred)                      # macro-averaged F1
n_correct = sum(r['correct'] for r in all_results)         # count correct predictions

print(f'Accuracy  : {acc:.2%}   ({n_correct}/{len(all_results)} correct)')
print(f'Macro F1  : {f1:.4f}')
print()

# ── Surface the failures ──────────────────────────────────────────────────
failures = [r for r in all_results if not r['correct']]    # filter to wrong predictions only
print(f'=== {len(failures)} FAILURES (out of {len(all_results)}) ===')
print()

for item in failures:                                      # detail each failure explicitly
    fid   = item['id']                                     # review ID
    ftrue = item['label']                                  # true label
    fpred = item['predicted']                              # predicted label
    ftext = item['text']                                   # verbatim review text

    print(f'Review {fid}')                                 # identify the review
    print(f'  True label : {ftrue}')                       # what the label says
    print(f'  Predicted  : {fpred}')                       # what the model returned
    print(f'  Text       : {ftext}')                       # the verbatim text
    print()

print('Discussion:')
print('  Both failures are NEGATIVE reviews predicted as POSITIVE.')
print('  Review 11: "arrived" biases the model toward positive despite "damaged".')
print('  Review 13: "would not recommend" is not parsed as a negated recommendation.')
print()
print('  Headline accuracy (86.67%) looks acceptable — but without surfacing')
print('  these two specific failures, the report cannot drive any improvement.')

Accuracy  : 86.67%   (13/15 correct)
Macro F1  : 0.8917

=== 2 FAILURES (out of 15) ===

Review 11
  True label : negative
  Predicted  : positive
  Text       : Item arrived damaged, very frustrating.

Review 13
  True label : negative
  Predicted  : positive
  Text       : Would not recommend to anyone.

Discussion:
  Both failures are NEGATIVE reviews predicted as POSITIVE.
  Review 11: "arrived" biases the model toward positive despite "damaged".
  Review 13: "would not recommend" is not parsed as a negated recommendation.

  Headline accuracy (86.67%) looks acceptable — but without surfacing
  these two specific failures, the report cannot drive any improvement.


In [22]:
# ── Shut down the background server ──────────────────────────────────────
server.terminate()                                         # send SIGTERM to the uvicorn process
#@time.sleep(0.5)                                           # allow the process a moment to exit cleanly
print(f'Server (PID {server.pid}) stopped.')              # confirm shutdown

Server (PID 583662) stopped.
